# CIPHER - separate probe budget ablation

Re-runs a model on a fixed 1/3 subset of CIPHER (333 instances, same ids/seeds as the
original runs) with **probes decoupled from the plan budget**: up to 3 `observe` probes
that do not change or advance the system and do not count against the 7-action plan.

Scoring: `cipher/separate_budget.py` (`separate-budget-v1`). The analysis re-scores
from the raw replies saved by this notebook, so the in-notebook score is only a
progress indicator.

**Before running:** upload a new version of the Kaggle dataset that contains
`cipher/separate_budget.py` and `data/instances_separate_budget.jsonl`.

Output: `/kaggle/working/separate_budget__<model>.jsonl` - one line per instance with the
raw reply (failures included). Download these into `results/separate_budget/`.

In [ ]:
# 1. Install dependencies
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "kaggle-benchmarks", "pandas"], check=True)

In [ ]:
# 2. Imports and paths
import os, sys, json, datetime, traceback
import pandas as pd
import kaggle_benchmarks as kbench

_KAGGLE_ROOT = "/kaggle/input/datasets/wenhaolu49/cipher"
CIPHER_ROOT = next((c for c in [_KAGGLE_ROOT, os.getcwd(), os.path.abspath("..")]
                    if os.path.isfile(os.path.join(c, "cipher", "separate_budget.py"))), None)
if CIPHER_ROOT is None:
    raise RuntimeError("cipher/separate_budget.py not found - attach the UPDATED cipher dataset version.")
sys.path.insert(0, CIPHER_ROOT)

from cipher.generator import Instance
from cipher.world import World, State, EntityState, Rule, Trigger, Effect
from cipher.schema import validate_response
from cipher.separate_budget import score_response_separate, SCORER_VERSION, PROBE_CAP
print("cipher root:", CIPHER_ROOT, "| scorer:", SCORER_VERSION, "| probe cap:", PROBE_CAP)

In [ ]:
# 3. Load the separate-budget instance file (UTF-8 explicitly; guard against mojibake)
DATA_PATH = os.path.join(CIPHER_ROOT, "data", "instances_separate_budget.jsonl")
with open(DATA_PATH, encoding="utf-8") as f:
    RECORDS = [json.loads(line) for line in f if line.strip()]
for r in RECORDS:
    assert "\u00e2\u20ac" not in r["prompt"] and "\u00ce\u00b1" not in r["prompt"], f"mojibake in {r['id']}"
    assert "do NOT consume this budget" in r["prompt"], f"not a separate-budget prompt: {r['id']}"
LIMIT = None  # set e.g. 5 for a smoke test
RECORDS = RECORDS[:LIMIT] if LIMIT else RECORDS
print(f"Loaded {len(RECORDS)} instances")
print("Available models:", list(kbench.llms.keys()))

In [ ]:
# 4. Parsing + scoring (same tolerant JSON extraction as the original benchmark)
import re as _re

def _extract_outermost_json(text):
    start = text.find("{")
    if start == -1:
        return ""
    depth, in_string, escape = 0, False, False
    for i, ch in enumerate(text[start:], start):
        if escape: escape = False; continue
        if ch == "\\" and in_string: escape = True; continue
        if ch == '"': in_string = not in_string; continue
        if in_string: continue
        if ch == "{": depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start:i + 1]
    return text[start:]

def _parse_json_tolerant(text):
    cleaned = _re.sub(r'//[^\n]*', '', text)
    cleaned = _re.sub(r',\s*([}\]])', r'\1', cleaned)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return json.loads(cleaned.replace("'", '"'))

def _instance_from_record(rec):
    h = rec["hidden"]
    rules = [Rule(name=r["name"],
                  trigger=Trigger(kind=r["trigger"]["kind"], i=r["trigger"]["i"], j=r["trigger"].get("j", -1), k=r["trigger"].get("k", 0)),
                  effect=Effect(kind=r["effect"]["kind"], target=r["effect"]["target"], delta=r["effect"].get("delta", 0), source=r["effect"].get("source", -1)))
             for r in h["rules"]]
    initial = State(tuple(EntityState(phase=s["phase"], flux=s["flux"]) for s in h["initial_state"]))
    return Instance(id=rec["id"], seed=rec["seed"], difficulty=rec["difficulty"],
                    world=World(initial=initial, rules=tuple(rules), horizon=h["horizon"]),
                    visible_rule_indices=h["visible_rule_indices"], hidden_rule_indices=h["hidden_rule_indices"],
                    public_rule_descriptions=[], hidden_fields=h.get("hidden_fields", []),
                    metacog_ground_truth=h["metacog_ground_truth"], true_unknown_ranking=h["true_unknown_ranking"],
                    oracle_objective=h.get("oracle_best"))

def score_reply(reply, rec):
    blob = _extract_outermost_json(reply)
    if not blob:
        return None, "no_json"
    try:
        raw = _parse_json_tolerant(blob)
    except Exception as e:
        return None, f"json_error: {e}"[:300]
    try:
        resp = validate_response(raw)
        sc = score_response_separate(resp, _instance_from_record(rec),
                                     best_obj=rec["hidden"]["oracle_best"], worst_obj=rec["hidden"]["oracle_worst"])
        return sc.to_dict(), "ok"
    except Exception as e:
        return None, f"score_error: {e}"[:300]

In [ ]:
# 5. Per-instance task: returns the RAW reply so nothing is lost on failure
@kbench.task(name="cipher_separate_budget_single", store_task=False)
def cipher_separate_budget_single(llm, prompt: str, record_id: str) -> dict:
    try:
        reply = str(llm.prompt(prompt))
        return {"record_id": record_id, "reply": reply, "error": None}
    except Exception as e:
        return {"record_id": record_id, "reply": None, "error": f"{type(e).__name__}: {e}"[:500]}

In [ ]:
# 6. Batch run: save raw replies + scores for every instance (failures included)
_LAST = {}

@kbench.task(name="cipher_separate_budget")
def cipher_separate_budget(llm):
    """CIPHER ablation: probes do not consume the plan budget."""
    df = pd.DataFrame([{"prompt": r["prompt"], "record_id": r["id"]} for r in RECORDS])
    with kbench.client.enable_cache():
        runs = cipher_separate_budget_single.evaluate(llm=[llm], evaluation_data=df, n_jobs=8, remove_run_files=True)
    results = runs.as_dataframe()["result"].tolist()

    rec_by_id = {r["id"]: r for r in RECORDS}
    model_tag = str(getattr(llm, "model_id", getattr(llm, "name", "unknown")))
    out_path = os.path.join("/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd(),
                            f"separate_budget__{model_tag.replace('/', '-')}.jsonl")
    rows, n_ok = [], 0
    for rec, res in zip(RECORDS, results):
        res = res if isinstance(res, dict) else {"record_id": rec["id"], "reply": None, "error": f"no result: {res!r}"[:300]}
        rid = res.get("record_id") or rec["id"]
        reply = res.get("reply")
        scores, status = (score_reply(reply, rec_by_id[rid]) if reply is not None else (None, "api_error"))
        n_ok += status == "ok"
        rows.append({"model": model_tag, "id": rid, "difficulty": rec_by_id[rid]["difficulty"],
                     "condition": "separate_probe_budget", "scorer_version": SCORER_VERSION,
                     "status": status, "error": res.get("error"), "reply": reply, "scores": scores,
                     "timestamp": datetime.datetime.utcnow().isoformat() + "Z"})
    with open(out_path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    ok = [r["scores"] for r in rows if r["scores"]]
    mean = lambda k: sum(s[k] for s in ok) / max(len(ok), 1)
    _LAST.update(rows=rows, out_path=out_path, n_ok=n_ok)
    print(f"{model_tag}: {n_ok}/{len(rows)} parsed -> {out_path}")
    for k in ["composite", "objective", "calibration", "attention", "executive", "A", "B", "n_probes_used"]:
        print(f"  {k:<14} {mean(k):.3f}")
    return mean("composite")

In [ ]:
%choose cipher_separate_budget
cipher_separate_budget.run(kbench.llm)